# 🎙️ WHISPER LARGE-V3 GPU SERVER - GOOGLE COLAB (TESLA T4 16GB)

Server xử lý âm thanh sang phụ đề CapCut siêu tốc sử dụng **Faster-Whisper Large-v3** kết hợp **Batched Inference** khai thác tối đa **13.0 - 13.5 GB VRAM** của GPU Tesla T4.

### 📋 Hướng dẫn sử dụng:
1. Chọn menu: **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** -> Bấm **Save**.
2. Bấm nút **Play (Run)** tại 2 ô bên dưới.
3. Chờ hiện dòng chữ: `👉 DÁN LINK NÀY VÀO TOOL DESKTOP: https://xxxx.trycloudflare.com`
4. Copy link đó và dán vào phần mềm CapCut STT Tool trên máy tính của bạn.

In [ ]:
# ==============================================================================
# BƯỚC 1: Cài đặt thư viện Faster-Whisper, FastAPI & Cloudflared
# ==============================================================================
!pip install -q faster-whisper "av<18" nvidia-cublas-cu12 nvidia-cudnn-cu12 fastapi uvicorn pydantic nest-asyncio soundfile
!python3 -c "import site, os; sp=site.getsitepackages()[0]; open('/etc/ld.so.conf.d/nvidia-cu12.conf', 'w').write(f'{sp}/nvidia/cublas/lib\n{sp}/nvidia/cudnn/lib\n'); os.system('ldconfig')"
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ Đã cài đặt hoàn tất các gói bổ trợ!")

In [ ]:
# ==============================================================================
# BƯỚC 2: Khởi chạy Server Whisper Large-v3 & Mở đường hầm Cloudflare
# ==============================================================================
import os
import io
import time
import base64
import tempfile
import threading
import subprocess
import re
import uuid
import gc
from typing import Optional

# Cấu hình bộ nhớ chống phân mảnh VRAM trên Tesla T4
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
# Cấu hình và nạp thư viện CUDA 12 (libcublas, libcudnn) cho CTranslate2
import glob, site, ctypes
for sp in site.getsitepackages():
    cublas_lib = os.path.join(sp, "nvidia", "cublas", "lib")
    cudnn_lib = os.path.join(sp, "nvidia", "cudnn", "lib")
    if os.path.exists(cublas_lib):
        os.environ["LD_LIBRARY_PATH"] = f"{cublas_lib}:{cudnn_lib}:" + os.environ.get("LD_LIBRARY_PATH", "")
        for lib_file in sorted(glob.glob(os.path.join(cublas_lib, "*.so*")) + glob.glob(os.path.join(cudnn_lib, "*.so*"))):
            try:
                ctypes.CDLL(lib_file, mode=ctypes.RTLD_GLOBAL)
            except Exception:
                pass

# Vá lỗi tương thích giữa PyAV và faster-whisper
try:
    import av
    _orig_av_open = av.open
    def _safe_av_open(*args, metadata_errors=None, **kwargs):
        return _orig_av_open(*args, **kwargs)
    av.open = _safe_av_open
except Exception:
    pass

from faster_whisper import WhisperModel, BatchedInferencePipeline

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"
BATCH_SIZE = 24  # Batching 24 câu đồng thời giúp đẩy VRAM lên 13.0 - 13.5 GB
BEAM_SIZE = 5    # SOTA accuracy cho tiếng Việt

print(f"--> Đang nạp model Whisper Large-v3 vào GPU ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})...")
base_model = WhisperModel(
    "large-v3",
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
    cpu_threads=4,
    num_workers=2
)

try:
    batched_pipeline = BatchedInferencePipeline(model=base_model)
    print("--> Kích hoạt BatchedInferencePipeline (Batching song song) thành công!")
except Exception as e:
    print(f"--> Dùng pipeline chuẩn do: {e}")
    batched_pipeline = None

app = FastAPI(title="Whisper Large-v3 Colab Server")

class TranscribeRequest(BaseModel):
    audio_base64: str
    audio_ext: Optional[str] = "flac"
    language: Optional[str] = "vi"
    batch_size: Optional[int] = 24
    beam_size: Optional[int] = 5
    vad_filter: Optional[bool] = True

TASKS = {}
TASKS_LOCK = threading.Lock()

def _execute_transcribe(req: TranscribeRequest):
    raw_bytes = base64.b64decode(req.audio_base64)
    ext = req.audio_ext or "flac"
    tmp_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix=f".{ext}", delete=False) as f:
            f.write(raw_bytes)
            tmp_path = f.name

        target_lang = None if (not req.language or req.language.lower() in ("auto", "none")) else req.language
        batch_size = req.batch_size or BATCH_SIZE
        beam_size = req.beam_size or BEAM_SIZE
        vad = True if req.vad_filter is None else req.vad_filter

        start_t = time.time()
        if batched_pipeline is not None and DEVICE == "cuda":
            segments_gen, info = batched_pipeline.transcribe(
                tmp_path,
                batch_size=batch_size,
                beam_size=beam_size,
                language=target_lang,
                vad_filter=vad,
                vad_parameters=dict(min_silence_duration_ms=500),
                word_timestamps=True
            )
        else:
            segments_gen, info = base_model.transcribe(
                tmp_path,
                beam_size=beam_size,
                language=target_lang,
                vad_filter=vad,
                vad_parameters=dict(min_silence_duration_ms=500),
                word_timestamps=True
            )

        parsed_segments = []
        for s in segments_gen:
            parsed_segments.append({
                "start": round(float(s.start), 3),
                "end": round(float(s.end), 3),
                "text": s.text.strip()
            })

        return {
            "success": True,
            "language": info.language,
            "processing_time_sec": round(time.time() - start_t, 2),
            "total_segments": len(parsed_segments),
            "segments": parsed_segments
        }
    finally:
        if tmp_path and os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except Exception:
                pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def _async_worker(task_id: str, req: TranscribeRequest):
    try:
        res = _execute_transcribe(req)
        with TASKS_LOCK:
            TASKS[task_id] = {"status": "completed", "result": res}
    except Exception as e:
        with TASKS_LOCK:
            TASKS[task_id] = {"status": "failed", "error": str(e)}
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

@app.get("/")
@app.get("/health")
def health_check():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    free_mem, total_mem = torch.cuda.mem_get_info(0) if torch.cuda.is_available() else (0, 0)
    used_mem = total_mem - free_mem
    return {
        "status": "ready",
        "gpu_name": gpu_name,
        "vram_used_gb": round(used_mem / (1024**3), 2),
        "vram_free_gb": round(free_mem / (1024**3), 2),
        "vram_total_gb": round(total_mem / (1024**3), 2),
        "model": "faster-whisper large-v3",
        "batch_size_target": BATCH_SIZE
    }

@app.post("/submit_transcribe")
def submit_transcribe(req: TranscribeRequest):
    task_id = str(uuid.uuid4())
    with TASKS_LOCK:
        TASKS[task_id] = {"status": "processing", "start_time": time.time()}
    threading.Thread(target=_async_worker, args=(task_id, req), daemon=True).start()
    return {"success": True, "task_id": task_id}

@app.get("/task_status/{task_id}")
def get_task_status(task_id: str):
    with TASKS_LOCK:
        task = TASKS.get(task_id)
    if not task:
        return {"status": "failed", "error": "Task not found"}
    if task["status"] == "completed":
        return {"status": "completed", "result": task["result"]}
    elif task["status"] == "failed":
        return {"status": "failed", "error": task.get("error", "Unknown error")}
    else:
        return {"status": "processing"}

@app.post("/task_clear/{task_id}")
def clear_task_status(task_id: str):
    with TASKS_LOCK:
        TASKS.pop(task_id, None)
    return {"success": True}

@app.post("/transcribe")
def transcribe_direct(req: TranscribeRequest):
    try:
        return _execute_transcribe(req)
    except Exception as e:
        return {"success": False, "error": str(e)}

@app.post("/purge_cache")
def purge_gpu_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"success": True, "message": "Đã giải phóng sạch VRAM và Cache GPU."}

# Khởi động FastAPI & Cloudflare Tunnel
nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"), daemon=True).start()
time.sleep(2)

if os.path.exists("cloudflare.log"):
    try:
        os.remove("cloudflare.log")
    except Exception:
        pass

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--logfile", "cloudflare.log"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

public_url = None
print("\nĐang thiết lập đường hầm Cloudflare Tunnel...")
for _ in range(35):
    time.sleep(1)
    if os.path.exists("cloudflare.log"):
        with open("cloudflare.log", "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
            matches = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if matches:
                public_url = matches[0]
                break

if public_url:
    print("\n" + "=" * 70)
    print("🚀 KẾT NỐI GPU COLAB THÀNH CÔNG (TESLA T4 16GB - WHISPER LARGE-V3)!")
    print(f"👉 DÁN LINK NÀY VÀO TOOL DESKTOP: {public_url}")
    print("=" * 70 + "\n")
    print("⏳ Server đang chạy ngầm... Giữ tab Google Colab mở để duy trì GPU.")
    try:
        tunnel_proc.wait()
    except KeyboardInterrupt:
        print("Đã dừng server.")
else:
    print("❌ Lỗi: Không thể lấy được link Cloudflare. Vui lòng kiểm tra file cloudflare.log.")